In [5]:
!pip install -U torchao peft trl transformers accelerate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 71.8 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 kB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 61.5 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 29.8 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.14.0
    Uninstalling accelerate-1.14.0:
      Successfully uninstalled accelerate-1.14.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.16.1
    Uninstalling transformers-5.16.1:
      Successfully uninstalled transformers-5.16.1
  Attempting uninstall: peft
    Found existing installation: peft 0.20.0
    Uninstalling peft-0.20.0:
      Successfully uninstalled peft-0.20.0


In [6]:
!pip install -U --no-cache-dir torchao peft

In [7]:
import json

train_data = [
    {"messages": [{"role": "system", "content": "You are MiniMind, a helpful and lightweight personal AI assistant."}, {"role": "user", "content": "Hi"}, {"role": "assistant", "content": "Hello! I am MiniMind. How can I help you today?"}]},
    {"messages": [{"role": "system", "content": "You are MiniMind, a helpful and lightweight personal AI assistant."}, {"role": "user", "content": "Who made you?"}, {"role": "assistant", "content": "I was trained and fine-tuned by my creator as the MiniMind model."}]},
    {"messages": [{"role": "system", "content": "You are MiniMind, a helpful and lightweight personal AI assistant."}, {"role": "user", "content": "What is Python?"}, {"role": "assistant", "content": "Python is a high-level, interpreted programming language known for its clear syntax and versatility in web development, machine learning, and automation."}]}
]

with open("train.jsonl", "w", encoding="utf-8") as f:
    for item in train_data:
        f.write(json.dumps(item) + "\n")

print("Dataset created: train.jsonl")

Dataset created: train.jsonl


In [8]:
import os
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
OUTPUT_DIR = "final_adapter"

# Load tokenizer and base model in float16 for fast GPU compute
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto"
)

dataset = load_dataset("json", data_files={"train": "train.jsonl"})

peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

training_args = SFTConfig(
    output_dir="checkpoints",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    num_train_epochs=3,
    logging_steps=1,
    save_strategy="no",
    eval_strategy="no",
    fp16=True,
    max_length=128
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    peft_config=peft_config,
    processing_class=tokenizer,
    args=training_args
)

print("Starting GPU Training...")
trainer.train()

# Save final adapter files
os.makedirs(OUTPUT_DIR, exist_ok=True)
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Training completed! Saved in {OUTPUT_DIR}")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

Tokenizing train dataset:   0%|          | 0/3 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/3 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/3 [00:00<?, ? examples/s]

...

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Starting GPU Training...


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
1,4.597260
2,4.169386
3,3.957013


Training completed! Saved in final_adapter


In [12]:
import os
import shutil

# 1. Zip the adapter directory
shutil.make_archive("final_adapter", "zip", "final_adapter")

# 2. Check and notify
if os.path.exists("final_adapter.zip"):
    print("\n" + "="*50)
    print("Download completed!")
    print("Please check your Colab folder (final_adapter.zip is ready).")
    print("="*50)
else:
    print("Error: Could not create zip file.")


Download completed!
Please check your Colab folder (final_adapter.zip is ready).
